<center><h1 style="margin: 1.5em 0;">Mathematical Explanation of the MNISTNet Implementation</h1></center>



A 3-layer fully connected neural network (784 → 256 → 128 → 10) trained with **mini-batch gradient descent**, **ReLU** hidden activations, **softmax** output, **cross-entropy loss** + **L2 regularization**.

## 1. Notation

- $X \in \mathbb{R}^{m \times 784}$ : mini-batch of flattened images  
- $Y \in \mathbb{R}^{m \times 10}$ : one-hot encoded true labels  
- $m$ : batch size  
- $W^{(l)} \in \mathbb{R}^{n_l \times n_{l-1}}$ : weight matrix of layer $l$  
- $b^{(l)} \in \mathbb{R}^{1 \times n_l}$ : bias vector  
- $Z^{(l)}, A^{(l)} \in \mathbb{R}^{m \times n_l}$ : pre- and post-activation of layer $l$  
- $A^{(0)} := X$

Layers:  
- $l=1$: 784 → 256  
- $l=2$: 256 → 128  
- $l=3$: 128 → 10 (output)

## 2. Forward Pass

$$
\begin{aligned}
Z^{(1)} &= X W^{(1)\top} + b^{(1)} \\
A^{(1)} &= \operatorname{ReLU}(Z^{(1)}) && \text{where } \operatorname{ReLU}(z) = \max(0,z) \\[1em]
Z^{(2)} &= A^{(1)} W^{(2)\top} + b^{(2)} \\
A^{(2)} &= \operatorname{ReLU}(Z^{(2)}) \\[1em]
Z^{(3)} &= A^{(2)} W^{(3)\top} + b^{(3)} \\
A^{(3)} &= \operatorname{softmax}(Z^{(3)})
\end{aligned}
$$

**Stable softmax** (used in code):

$$
\operatorname{softmax}(z_i) = \frac{\exp(z_i - \max_j z_j)}{\sum_k \exp(z_k - \max_j z_j)}
$$

$A^{(3)}$ contains predicted probabilities $\hat{y}_i \in [0,1]^{10}$ with $\sum_c \hat{y}_{i,c} = 1$.

## 3. Loss Function

**Per mini-batch loss** (average cross-entropy + L2 regularization):

$$
\mathcal{L} = -\frac{1}{m} \sum_{i=1}^m \sum_{c=1}^{10} y_{i,c} \log(\hat{y}_{i,c}) + \frac{\lambda}{2m} \sum_{l=1}^3 \|W^{(l)}\|_F^2
$$

- First term = categorical cross-entropy (CE)  
- Second term = weight decay (L2 regularization)  
- $\lambda$ = regularization strength (`reg` in code)

## 4. Backpropagation – Gradient Computation

### Output layer (l = 3)

$$
\frac{\partial \mathcal{L}}{\partial Z^{(3)}} = A^{(3)} - Y \qquad \text{(very convenient property of softmax + CE)}
$$

$$
\frac{\partial \mathcal{L}}{\partial W^{(3)}} = \frac{1}{m} \left( {dZ^{(3)}}^\top A^{(2)} \right) + \frac{\lambda}{m} W^{(3)}
$$

$$
\frac{\partial \mathcal{L}}{\partial b^{(3)}} = \frac{1}{m} \sum_{i=1}^m dZ^{(3)}_{i,:} \quad \text{(row vector)}
$$

### Hidden layers (l = 2 and l = 1)

$$
dZ^{(l)} = \left( dZ^{(l+1)} W^{(l+1)} \right) \odot \mathbb{1}_{\{Z^{(l)} > 0\}}
$$

$$
\frac{\partial \mathcal{L}}{\partial W^{(l)}} = \frac{1}{m} \left( {dZ^{(l)}}^\top A^{(l-1)} \right) + \frac{\lambda}{m} W^{(l)}
$$

$$
\frac{\partial \mathcal{L}}{\partial b^{(l)}} = \frac{1}{m} \sum_{i=1}^m dZ^{(l)}_{i,:}
$$

(where $A^{(0)} = X$ and $\odot$ is element-wise multiplication)

## 5. Parameter Update (Vanilla Gradient Descent)

$$
W^{(l)} \leftarrow W^{(l)} - \eta \cdot \frac{\partial \mathcal{L}}{\partial W^{(l)}}, \quad
b^{(l)} \leftarrow b^{(l)} - \eta \cdot \frac{\partial \mathcal{L}}{\partial b^{(l)}}
$$

$\eta =$ learning rate (`lr` in code)

## 6. Weight Initialization – He initialization (for ReLU)

$$
W^{(l)}_{ij} \sim \mathcal{N}\left(0, \sqrt{\frac{2}{n_{l-1}}}\right)
$$

This scaling helps keep the variance of activations roughly constant across layers when using ReLU.

## Summary Table

| Layer | Input dim | Output dim | Activation | Gradient w.r.t. Z          |
|-------|-----------|------------|------------|-----------------------------|
| 1     | 784       | 256        | ReLU       | $(dZ^{(2)} W^{(2)}) \odot \mathbb{1}_{Z^{(1)}>0}$ |
| 2     | 256       | 128        | ReLU       | $(dZ^{(3)} W^{(3)}) \odot \mathbb{1}_{Z^{(2)}>0}$ |
| 3     | 128       | 10         | softmax    | $A^{(3)} - Y$               |



In [1]:
import numpy as np
from tensorflow.keras.datasets import mnist

In [2]:
class MNISTNet:
    def __init__(self, lr=0.05, reg=1e-4):
        self.lr = lr
        self.reg = reg
        self._init_params()

    def _init_params(self):
        dims = [784, 256, 128, 10]
        self.params = {}

        for i in range(1, len(dims)):
            self.params[f"W{i}"] = (
                np.random.randn(dims[i], dims[i-1]) *
                np.sqrt(2.0 / dims[i-1])
            )
            self.params[f"b{i}"] = np.zeros((1, dims[i]))  # row bias

    # --------------------------------------------------
    # ACTIVATIONS
    # --------------------------------------------------

    def relu(self, Z):
        return np.maximum(0, Z)

    def relu_grad(self, Z):
        return (Z > 0).astype(float)

    def softmax(self, Z):
        Z_stable = Z - np.max(Z, axis=1, keepdims=True)
        exp = np.exp(Z_stable)
        return exp / np.sum(exp, axis=1, keepdims=True)

    # --------------------------------------------------
    # FORWARD
    # --------------------------------------------------

    def forward(self, X):
        self.cache = {}

        W1, b1 = self.params["W1"], self.params["b1"]
        W2, b2 = self.params["W2"], self.params["b2"]
        W3, b3 = self.params["W3"], self.params["b3"]

        # (m,784)(784,256) = (m,256)
        Z1 = X @ W1.T + b1
        A1 = self.relu(Z1)

        Z2 = A1 @ W2.T + b2
        A2 = self.relu(Z2)

        Z3 = A2 @ W3.T + b3
        A3 = self.softmax(Z3)

        self.cache = {
            "X": X,
            "Z1": Z1, "A1": A1,
            "Z2": Z2, "A2": A2,
            "Z3": Z3, "A3": A3
        }

        return A3

    # --------------------------------------------------
    # LOSS
    # --------------------------------------------------

    def compute_loss(self, Y_hat, Y):
        m = Y.shape[0]

        ce = -np.sum(Y * np.log(Y_hat + 1e-8)) / m

        l2 = sum(np.sum(self.params[f"W{i}"]**2) for i in range(1,4))
        l2 = (self.reg / (2*m)) * l2

        return ce + l2

    # --------------------------------------------------
    # BACKPROP
    # --------------------------------------------------

    def backward(self, Y):
        m = Y.shape[0]
        grads = {}

        X  = self.cache["X"]
        A1 = self.cache["A1"]
        A2 = self.cache["A2"]
        A3 = self.cache["A3"]
        Z1 = self.cache["Z1"]
        Z2 = self.cache["Z2"]

        W1 = self.params["W1"]
        W2 = self.params["W2"]
        W3 = self.params["W3"]

        # (m,10)
        dZ3 = A3 - Y

        grads["W3"] = (dZ3.T @ A2) / m + (self.reg/m)*W3
        grads["b3"] = np.sum(dZ3, axis=0, keepdims=True) / m

        dZ2 = (dZ3 @ W3) * self.relu_grad(Z2)

        grads["W2"] = (dZ2.T @ A1) / m + (self.reg/m)*W2
        grads["b2"] = np.sum(dZ2, axis=0, keepdims=True) / m

        dZ1 = (dZ2 @ W2) * self.relu_grad(Z1)

        grads["W1"] = (dZ1.T @ X) / m + (self.reg/m)*W1
        grads["b1"] = np.sum(dZ1, axis=0, keepdims=True) / m

        return grads

    # --------------------------------------------------
    # UPDATE
    # --------------------------------------------------

    def update(self, grads):
        for i in range(1,4):
            self.params[f"W{i}"] -= self.lr * grads[f"W{i}"]
            self.params[f"b{i}"] -= self.lr * grads[f"b{i}"]

    # --------------------------------------------------
    # TRAIN
    # --------------------------------------------------

    def train(self, X_train, Y_train, X_test, Y_test,
              epochs=20, batch_size=128):

        m = X_train.shape[0]

        for epoch in range(1, epochs+1):
            idx = np.random.permutation(m)

            for i in range(0, m, batch_size):
                batch_idx = idx[i:i+batch_size]

                X_batch = X_train[batch_idx]
                Y_batch = Y_train[batch_idx]

                Y_hat = self.forward(X_batch)
                grads = self.backward(Y_batch)
                self.update(grads)

            train_acc = self.accuracy(X_train, Y_train)
            test_acc  = self.accuracy(X_test, Y_test)
            loss = self.compute_loss(self.forward(X_train), Y_train)

            print(f"Epoch {epoch:2d} | "
                  f"Loss {loss:.4f} | "
                  f"Train {train_acc:.2f}% | "
                  f"Test {test_acc:.2f}%")

    def predict(self, X):
        return np.argmax(self.forward(X), axis=1)

    def accuracy(self, X, Y):
        preds = self.predict(X)
        labels = np.argmax(Y, axis=1)
        return np.mean(preds == labels) * 100

In [3]:
import numpy as np
from tensorflow.keras.datasets import mnist

print("Loading MNIST...")
(X_train, Y_train), (X_test, Y_test) = mnist.load_data()

# Flatten images (keep 60000, 784 format)
X_train = X_train.reshape(60000, -1) / 255.0
X_test  = X_test.reshape(10000, -1)  / 255.0

print("Train shape:", X_train.shape)
print("Test shape :", X_test.shape)

Loading MNIST...
11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Train shape: (60000, 784)
Test shape : (10000, 784)


In [4]:
def one_hot(Y, num_classes=10):
    return np.eye(num_classes)[Y]

Y_train_oh = one_hot(Y_train)
Y_test_oh  = one_hot(Y_test)

print("Y_train shape:", Y_train_oh.shape)

Y_train shape: (60000, 10)


In [5]:
net = MNISTNet(lr=0.05, reg=1e-4)

net.train(
    X_train, Y_train_oh,
    X_test,  Y_test_oh,
    epochs=20,
    batch_size=128
)

Epoch  1 | Loss 0.3018 | Train 91.27% | Test 91.65%
Epoch  2 | Loss 0.2078 | Train 94.19% | Test 94.08%
Epoch  3 | Loss 0.1675 | Train 95.20% | Test 94.96%
Epoch  4 | Loss 0.1429 | Train 96.00% | Test 95.60%
Epoch  5 | Loss 0.1234 | Train 96.54% | Test 96.03%
Epoch  6 | Loss 0.1081 | Train 96.98% | Test 96.59%
Epoch  7 | Loss 0.0975 | Train 97.26% | Test 96.76%
Epoch  8 | Loss 0.0885 | Train 97.52% | Test 96.89%
Epoch  9 | Loss 0.0796 | Train 97.84% | Test 97.00%
Epoch 10 | Loss 0.0738 | Train 97.88% | Test 97.09%
Epoch 11 | Loss 0.0642 | Train 98.29% | Test 97.32%
Epoch 12 | Loss 0.0602 | Train 98.31% | Test 97.29%
Epoch 13 | Loss 0.0576 | Train 98.41% | Test 97.27%
Epoch 14 | Loss 0.0497 | Train 98.69% | Test 97.54%
Epoch 15 | Loss 0.0506 | Train 98.61% | Test 97.36%
Epoch 16 | Loss 0.0427 | Train 98.92% | Test 97.62%
Epoch 17 | Loss 0.0406 | Train 98.97% | Test 97.61%
Epoch 18 | Loss 0.0385 | Train 99.04% | Test 97.73%
Epoch 19 | Loss 0.0333 | Train 99.21% | Test 97.70%
Epoch 20 | L